In [67]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np 

In [68]:
df = pd.read_csv('../data_sets/czyste_dane')
# zmienna do przewidywania cena aktualna

In [69]:
df.head(5)

,kategoria_produktu,nazwa,marka,cena_aktualna,cena_regularna,material,fason,ksztalt,rodzaj_dekoltu,dlugosc_rekawa,fason_clean,trend_score
0,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,nike sportswear,199.00,179.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
1,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,karl lagerfeld,265.28,NaN,"95% bawełna, 5% elastan",regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
2,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,iceberg,899.00,779.00,100% bawełna,regular,Prosty,Okrągły,Krótki rękaw,koszulka regular fit,3.33
3,koszulki z krótkim rękawem (męskie),- t-shirt z nadrukiem,dc shoes,101.99,169.99,100% bawełna,loose fit,Prosty,Okrągły,Krótki rękaw,koszulka loose fit,3.33
4,koszulki z krótkim rękawem (męskie),- bluzka z długim rękawem,adidas originals,197.00,359.00,100% poliester,regular,Prosty,NaN,Długi rękaw,koszulka regular fit,3.33


In [70]:
import re
import html
import pandas as pd

def clean_nazwa(value):
    if pd.isna(value):
        return None

    x = html.unescape(str(value)).lower().strip()

    # usuń początkowe "-"
    x = re.sub(r'^\s*-\s*', '', x)

    # usuń packi (2 pack, 3-pack, 2pk itd.)
    x = re.sub(r'\b\d+\s*-?\s*pack\b', '', x)
    x = re.sub(r'\b\d+er-?pack\b', '', x)
    x = re.sub(r'\btee\s*\d+\s*-?\s*pack\b', '', x)
    x = re.sub(r'\b\d+\s*pk\b', '', x)

    # czyszczenie spacji
    x = re.sub(r'\s+', ' ', x).strip()

    # jeśli puste
    if x in ['', '-', '–']:
        return None

    # MAPOWANIE NA KATEGORIE
    patterns = {
        "t-shirt z nadrukiem": "t-shirt z nadrukiem",
        "t-shirt basic": "t-shirt basic",
        "koszulka polo": "koszulka polo",
        "bluzka z długim rękawem": "bluzka z długim rękawem",
        "koszulka sportowa": "koszulka sportowa",
        "koszulka reprezentacji": "koszulka reprezentacji",
        "top": "top",
        "artykuły klubowe": "artykuły klubowe",
        "bluза z kapturem": "bluza z kapturem",
        "spodnie treningowe": "spodnie treningowe",
        "szorty": None   # 🔥 WARUNEK: wyrzucamy szorty
    }

    for key, val in patterns.items():
        if key in x:
            return val

    return None

In [71]:
df['nazwa'] = df['nazwa'].apply(clean_nazwa)

In [72]:
df['nazwa'].unique().tolist()

['t-shirt z nadrukiem',
 'bluzka z długim rękawem',
 't-shirt basic',
 'koszulka polo',
 'artykuły klubowe',
 'koszulka sportowa',
 'top',
 nan,
 'koszulka reprezentacji',
 'spodnie treningowe']

In [73]:
df['nazwa'].isna().sum()

np.int64(3)

In [74]:
df['nazwa'].isna().sum()

np.int64(3)

In [75]:
df['marka'].unique().tolist()

['nike sportswear',
 'karl lagerfeld',
 'iceberg',
 'dc shoes',
 'adidas originals',
 'indicode jeans wilbur',
 'lyle & scott',
 'vilebrequin portisol',
 'edwin unisex',
 'tommy jeans',
 'liu jo',
 'massimo dutti',
 'calvin klein golf newport',
 'boggi milano',
 'thinking mu aaron',
 'pegador pike oversized',
 'next',
 'nike performance stride',
 'umbro unisex',
 'polo ralph lauren',
 'urban classics do not use',
 'quiksilver',
 's.oliver t',
 'u.s. polo assn. 3 pack',
 'jack & jones jormontauk',
 'lonsdale st. erney',
 'adidas performance',
 'kronstadt timmi',
 'puma',
 'jack & jones',
 'karl kani',
 'pier one',
 'lacoste',
 'nike golf',
 'hollister co.',
 'mister tee feel the heat round neck',
 'under armour',
 'lost youth',
 'clean cut copenhagen copenhagen',
 'lonsdale blairmore',
 'carhartt wip emerge',
 'mango',
 's.oliver',
 'jack & jones premium jprblamason',
 'wasted paris unisex',
 'yourturn unisex',
 'pepe jeans jacko',
 'tom tailor denim',
 'upscale by mister tee t',
 'rock

In [76]:
import re
import html
import pandas as pd

def clean_brand(value):
    if pd.isna(value):
        return None

    x = html.unescape(str(value)).lower().strip()
    x = re.sub(r"\s+", " ", x)

    # usuń packi
    x = re.sub(r"\b\d+\s*-?\s*pack\b", "", x)
    x = re.sub(r"\b\d+er-?pack\b", "", x)
    x = re.sub(r"\b\d+\s*pk\b", "", x)
    x = re.sub(r"\btee\s*\d+\s*-?\s*pack\b", "", x)
    x = re.sub(r"\s+", " ", x).strip()

    brand_patterns = [
        (r"^jack & jones premium", "jack & jones"),
        (r"^jack & jones", "jack & jones"),
        (r"^u\.s\. polo assn\.", "u.s. polo assn."),
        (r"^polo ralph lauren", "polo ralph lauren"),
        (r"^adidas originals", "adidas originals"),
        (r"^adidas performance", "adidas performance"),
        (r"^nike sportswear", "nike sportswear"),
        (r"^nike performance", "nike performance"),
        (r"^under armour", "under armour"),
        (r"^tommy hilfiger", "tommy hilfiger"),
        (r"^calvin klein jeans", "calvin klein"),
        (r"^calvin klein golf", "calvin klein"),
        (r"^calvin klein", "calvin klein"),
        (r"^hugo", "hugo"),
        (r"^boss", "boss"),
        (r"^puma", "puma"),
        (r"^lacoste sport", "lacoste"),
        (r"^lacoste", "lacoste"),
        (r"^guess jeans", "guess"),
        (r"^guess", "guess"),
        (r"^levi's® plus", "levi's®"),
        (r"^levi's®", "levi's®"),
        (r"^s\.oliver black label", "s.oliver"),
        (r"^s\.oliver", "s.oliver"),
        (r"^ea7 emporio armani", "ea7 emporio armani"),
        (r"^emporio armani", "emporio armani"),
        (r"^armani exchange", "armani exchange"),
        (r"^mister tee", "mister tee"),
        (r"^only & sons", "only & sons"),
        (r"^selected", "selected"),
        (r"^carhartt wip", "carhartt wip"),
        (r"^karl lagerfeld jeans", "karl lagerfeld"),
        (r"^karl lagerfeld", "karl lagerfeld"),
        (r"^the north face", "the north face"),
        (r"^new balance", "new balance"),
        (r"^polo club", "polo club"),
        (r"^versace jeans couture", "versace jeans couture"),
        (r"^versace", "versace"),
        (r"^marc o'polo denim", "marc o'polo"),
        (r"^marc o'polo", "marc o'polo"),
        (r"^o'neill", "o'neill"),
        (r"^gant", "gant"),
        (r"^joop! jeans", "joop!"),
        (r"^joop!", "joop!"),
        (r"^pepe jeans", "pepe jeans"),
        (r"^lonsdale", "lonsdale"),
        (r"^blend", "blend"),
        (r"^solid", "solid"),
        (r"^hummel", "hummel"),
        (r"^fila", "fila"),
        (r"^diesel", "diesel"),
        (r"^tom tailor denim", "tom tailor"),
        (r"^tom tailor", "tom tailor"),
        (r"^boggi milano", "boggi milano"),
        (r"^bogner", "bogner"),
        (r"^patagonia", "patagonia"),
        (r"^scotch & soda", "scotch & soda"),
        (r"^samsøe samsøe", "samsøe samsøe"),
        (r"^weekend offender", "weekend offender"),
        (r"^hollister co\.", "hollister co."),
        (r"^urban classics", "urban classics"),
        (r"^quiksilver", "quiksilver"),
        (r"^billabong", "billabong"),
        (r"^volcom", "volcom"),
        (r"^oakley", "oakley"),
        (r"^champion", "champion"),
        (r"^arena", "arena"),
        (r"^the kooples", "the kooples"),
    ]

    for pattern, canonical in brand_patterns:
        if re.match(pattern, x):
            return canonical

    parts = x.split()
    if not parts:
        return None

    # fallback: pierwsze 1-2 słowa
    if len(parts) >= 2:
        return " ".join(parts[:2])
    return parts[0]

In [77]:
df['marka'] = df['marka'].apply(clean_brand)

In [78]:
df['marka'].unique().tolist()

['nike sportswear',
 'karl lagerfeld',
 'iceberg',
 'dc shoes',
 'adidas originals',
 'indicode jeans',
 'lyle &',
 'vilebrequin portisol',
 'edwin unisex',
 'tommy jeans',
 'liu jo',
 'massimo dutti',
 'calvin klein',
 'boggi milano',
 'thinking mu',
 'pegador pike',
 'next',
 'nike performance',
 'umbro unisex',
 'polo ralph lauren',
 'urban classics',
 'quiksilver',
 's.oliver',
 'u.s. polo assn.',
 'jack & jones',
 'lonsdale',
 'adidas performance',
 'kronstadt timmi',
 'puma',
 'karl kani',
 'pier one',
 'lacoste',
 'nike golf',
 'hollister co.',
 'mister tee',
 'under armour',
 'lost youth',
 'clean cut',
 'carhartt wip',
 'mango',
 'wasted paris',
 'yourturn unisex',
 'pepe jeans',
 'tom tailor',
 'upscale by',
 'rockshirts',
 'blend',
 'santa cruz',
 'tommy hilfiger',
 'billabong',
 'jp1880',
 'ea7 emporio armani',
 'michael kors',
 'selected',
 'henry tiger',
 'adidas golf',
 'bogner',
 'champion',
 'boss',
 'redefined rebel',
 'wood wood',
 'forsberg',
 'red bull',
 'seidenst

In [79]:
df['material'].unique().tolist()

['100% bawełna',
 '95% bawełna, 5% elastan',
 '100% poliester',
 '88% poliester, 12% elastan',
 '56% bawełna, 39% poliester, 5% elastan',
 '94% len, 6% elastan',
 '60% bawełna, 40% poliester',
 '83% poliester, 17% elastan',
 '52% bawełna, 48% poliester',
 '54% bawełna, 46% poliester',
 '65% bawełna, 35% poliester',
 '85% poliester, 15% elastan',
 '52% lyocell, 48% poliester',
 '60% modal, 23% poliamid, 17% jedwab',
 '92% bawełna, 8% elastan',
 '51% bawełna, 45% poliester, 4% elastan',
 '65% poliester, 35% bawełna',
 '94% poliester, 6% elastan',
 '53% bawełna, 47% wiskoza',
 '85% bawełna, 15% poliester',
 '50% bawełna, 48% poliester, 2% elastan',
 '60% bawełna, 20% wiskoza, 20% nylon',
 '58% len, 42% wiskoza',
 '78% bawełna, 15% poliester, 7% wiskoza',
 '50% poliester, 50% bawełna',
 '50% bawełna, 50% poliester',
 '56% wełna, 44% poliester',
 '40% poliester, 40% bawełna, 20% wiskoza',
 '80% bawełna, 20% wiskoza',
 '77% poliester, 16% wiskoza, 7% elastan',
 '90% poliester, 10% elastan',


In [80]:
import re
import pandas as pd

MATERIALS = [
    'bawełna',
    'poliester',
    'elastan',
    'wiskoza',
    'wełna',
    'len',
    'jedwab',
    'nylon',
    'poliamid',
    'lyocell',
    'modal',
    'akryl',
    'kaszmir',
    'poliuretan'
]

NATURAL_MATERIALS = [
    'bawełna',
    'wełna',
    'len',
    'jedwab',
    'kaszmir'
]

SYNTHETIC_MATERIALS = [
    'poliester',
    'elastan',
    'nylon',
    'poliamid',
    'akryl',
    'poliuretan'
]

PREMIUM_MATERIALS = [
    'kaszmir',
    'jedwab',
    'wełna',
    'len'
]


def extract_material_features(text):

    # procenty materiałów
    result = {
        f'{mat}_pct': 0 for mat in MATERIALS
    }

    result['main_material'] = None
    result['natural_material_pct'] = 0
    result['synthetic_material_pct'] = 0
    result['premium_material_pct'] = 0

    if pd.isna(text):
        return result

    text = str(text).lower()

    # znajdź np. "95% bawełna"
    matches = re.findall(
        r'(\d+)%\s*([a-ząćęłńóśźż]+)',
        text
    )

    max_pct = 0

    for pct, mat in matches:

        pct = int(pct)

        if mat in MATERIALS:

            result[f'{mat}_pct'] = pct

            # główny materiał
            if pct > max_pct:
                max_pct = pct
                result['main_material'] = mat

            # natural
            if mat in NATURAL_MATERIALS:
                result['natural_material_pct'] += pct

            # synthetic
            if mat in SYNTHETIC_MATERIALS:
                result['synthetic_material_pct'] += pct

            # premium
            if mat in PREMIUM_MATERIALS:
                result['premium_material_pct'] += pct

    return result

In [81]:
material_df = (
    df['material']
    .apply(extract_material_features)
    .apply(pd.Series)
)

df = pd.concat([df, material_df], axis=1)

In [82]:
# uzpełnianie braków
df['cena_regularna'] = df['cena_regularna'].fillna( df['cena_regularna'].median())
df['ksztalt'] = df['ksztalt'].fillna(df['ksztalt'].mode()[0])
df['rodzaj_dekoltu'] = df['rodzaj_dekoltu'].fillna(df['rodzaj_dekoltu'].mode()[0])
df['nazwa'] = df['nazwa'].fillna(df['nazwa'].mode()[0])

In [83]:
df.columns

Index(['kategoria_produktu', 'nazwa', 'marka', 'cena_aktualna',
       'cena_regularna', 'material', 'fason', 'ksztalt', 'rodzaj_dekoltu',
       'dlugosc_rekawa', 'fason_clean', 'trend_score', 'bawełna_pct',
       'poliester_pct', 'elastan_pct', 'wiskoza_pct', 'wełna_pct', 'len_pct',
       'jedwab_pct', 'nylon_pct', 'poliamid_pct', 'lyocell_pct', 'modal_pct',
       'akryl_pct', 'kaszmir_pct', 'poliuretan_pct', 'main_material',
       'natural_material_pct', 'synthetic_material_pct',
       'premium_material_pct'],
      dtype='str')

In [ ]:
df = df.drop(columns = ['material', 'main_material', 'rodzaj_dekoltu', 'fason', 'ksztalt'])

In [60]:
df.columns

Index(['kategoria_produktu', 'nazwa', 'marka', 'cena_aktualna',
       'cena_regularna', 'fason', 'ksztalt', 'dlugosc_rekawa', 'fason_clean',
       'trend_score', 'bawełna_pct', 'poliester_pct', 'elastan_pct',
       'wiskoza_pct', 'wełna_pct', 'len_pct', 'jedwab_pct', 'nylon_pct',
       'poliamid_pct', 'lyocell_pct', 'modal_pct', 'akryl_pct', 'kaszmir_pct',
       'poliuretan_pct', 'natural_material_pct', 'synthetic_material_pct',
       'premium_material_pct'],
      dtype='str')

In [66]:
df['ksztalt'].unique().tolist()

['Prosty', 'Dopasowany', 'Rozkloszowany', 'Dopasowanie do figury', 'Zwężany']

In [41]:
df.to_csv('final_czyste_dane', index = False)